In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Locate the project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

input_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_demographic_competition_metrics.csv"
)

tract_data = pd.read_csv(
    input_path,
    dtype={
        "GEOID": "string",
        "STATEFP": "string",
        "COUNTYFP": "string",
        "TRACTCE": "string"
    }
)

required_columns = [
    "GEOID",
    "total_population",
    "chinese_total_estimate",
    "chinese_total_moe",
    "chinese_share_pct",
    "chinese_estimate_reliability",
    "median_household_income",
    "competitors_within_1_mile",
    "competitors_within_3_miles"
]

missing_summary = (
    tract_data[required_columns]
    .isna()
    .sum()
    .rename_axis("column")
    .reset_index(name="missing_count")
)

print("Rows loaded:", len(tract_data))
print(
    "Unique GEOIDs:",
    tract_data["GEOID"].nunique()
)
print(
    "Duplicate GEOIDs:",
    tract_data["GEOID"].duplicated().sum()
)

display(missing_summary)

display(
    tract_data[
        [
            "total_population",
            "chinese_total_estimate",
            "chinese_share_pct",
            "median_household_income",
            "competitors_within_1_mile",
            "competitors_within_3_miles"
        ]
    ].describe()
    .round(2)
)

Rows loaded: 2498
Unique GEOIDs: 2498
Duplicate GEOIDs: 0


,column,missing_count
0,GEOID,0
1,total_population,0
2,chinese_total_estimate,0
3,chinese_total_moe,0
4,chinese_share_pct,21
5,chinese_estimate_reliability,0
6,median_household_income,42
7,competitors_within_1_mile,0
8,competitors_within_3_miles,0


,total_population,chinese_total_estimate,chinese_share_pct,median_household_income,competitors_within_1_mile,competitors_within_3_miles
count,2498.00,2498.00,2477.00,2456.00,2498.00,2498.00
mean,3926.61,190.27,4.74,95746.35,1.93,14.01
std,1425.77,427.02,9.86,40892.99,2.41,11.84
min,0.00,0.00,0.00,5213.00,0.00,0.00
25%,2969.50,1.00,0.06,66568.25,0.00,6.00
50%,3855.50,41.00,1.11,88571.00,1.00,11.00
75%,4818.50,151.75,4.21,115232.75,3.00,18.00
max,13681.00,3580.00,78.94,250001.00,16.00,52.00


In [2]:
# Create the preliminary analysis-eligible tract subset

accepted_reliability = [
    "more_reliable",
    "usable_with_caution"
]

eligible_tracts = tract_data.loc[
    tract_data["chinese_estimate_reliability"].isin(
        accepted_reliability
    )
    & tract_data["chinese_share_pct"].notna()
    & tract_data["median_household_income"].notna()
    & (tract_data["total_population"] > 0)
].copy()

print("All tracts:", len(tract_data))
print(
    "Eligible tracts for preliminary screening:",
    len(eligible_tracts)
)
print(
    "Excluded tracts:",
    len(tract_data) - len(eligible_tracts)
)

reliability_summary = (
    eligible_tracts[
        "chinese_estimate_reliability"
    ]
    .value_counts()
    .rename_axis("reliability")
    .reset_index(name="tract_count")
)

display(reliability_summary)

# Use rank-based Spearman correlation because variables are skewed
correlation_columns = [
    "chinese_total_estimate",
    "chinese_share_pct",
    "median_household_income",
    "competitors_within_1_mile",
    "competitors_within_3_miles"
]

spearman_correlation = (
    eligible_tracts[correlation_columns]
    .corr(method="spearman")
    .round(2)
)

display(spearman_correlation)

All tracts: 2498
Eligible tracts for preliminary screening: 964
Excluded tracts: 1534


,reliability,tract_count
0,usable_with_caution,665
1,more_reliable,299


,chinese_total_estimate,chinese_share_pct,median_household_income,competitors_within_1_mile,competitors_within_3_miles
chinese_total_estimate,1.00,0.94,-0.03,0.16,0.25
chinese_share_pct,0.94,1.00,-0.05,0.19,0.32
median_household_income,-0.03,-0.05,1.00,-0.46,-0.44
competitors_within_1_mile,0.16,0.19,-0.46,1.00,0.67
competitors_within_3_miles,0.25,0.32,-0.44,0.67,1.00


In [3]:
# Convert raw indicators to percentile scores

eligible_tracts["chinese_population_percentile"] = (
    eligible_tracts["chinese_total_estimate"]
    .rank(pct=True, method="average")
)

eligible_tracts["chinese_share_percentile"] = (
    eligible_tracts["chinese_share_pct"]
    .rank(pct=True, method="average")
)

eligible_tracts["income_percentile"] = (
    eligible_tracts["median_household_income"]
    .rank(pct=True, method="average")
)

eligible_tracts["low_competition_1_mile_percentile"] = (
    1
    - eligible_tracts["competitors_within_1_mile"]
    .rank(pct=True, method="average")
)

eligible_tracts["low_competition_3_miles_percentile"] = (
    1
    - eligible_tracts["competitors_within_3_miles"]
    .rank(pct=True, method="average")
)

# Combine closely related indicators into components

eligible_tracts["demand_component"] = (
    0.60
    * eligible_tracts["chinese_population_percentile"]
    + 0.40
    * eligible_tracts["chinese_share_percentile"]
)

eligible_tracts["competition_component"] = (
    0.40
    * eligible_tracts[
        "low_competition_1_mile_percentile"
    ]
    + 0.60
    * eligible_tracts[
        "low_competition_3_miles_percentile"
    ]
)

# Preliminary market opportunity score

eligible_tracts["preliminary_opportunity_score"] = (
    100
    * (
        0.60 * eligible_tracts["demand_component"]
        + 0.20 * eligible_tracts["income_percentile"]
        + 0.20 * eligible_tracts["competition_component"]
    )
).round(2)

ranking_columns = [
    "GEOID",
    "chinese_total_estimate",
    "chinese_share_pct",
    "median_household_income",
    "competitors_within_1_mile",
    "competitors_within_3_miles",
    "chinese_estimate_reliability",
    "demand_component",
    "competition_component",
    "preliminary_opportunity_score"
]

preliminary_ranking = (
    eligible_tracts[ranking_columns]
    .sort_values(
        "preliminary_opportunity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

preliminary_ranking.index = (
    preliminary_ranking.index + 1
)

display(
    preliminary_ranking.head(20)
    .round(
        {
            "demand_component": 3,
            "competition_component": 3
        }
    )
)

,GEOID,chinese_total_estimate,chinese_share_pct,median_household_income,competitors_within_1_mile,competitors_within_3_miles,chinese_estimate_reliability,demand_component,competition_component,preliminary_opportunity_score
1,06037403407,1661,70.14,156552.0,0,6,more_reliable,0.962,0.806,91.01
2,06037430400,1674,39.66,182292.0,0,8,more_reliable,0.936,0.744,89.53
3,06037464102,2617,57.62,239052.0,0,16,more_reliable,0.987,0.515,89.11
4,06037403325,2735,58.12,111445.0,0,2,more_reliable,0.989,0.893,89.06
5,06037403404,1035,48.96,173289.0,0,7,more_reliable,0.915,0.774,88.61
6,06037464101,1376,61.90,250001.0,0,13,more_reliable,0.946,0.593,88.52
7,06037670413,881,17.99,214625.0,0,0,more_reliable,0.837,0.929,88.16
8,06037408503,1943,29.42,145921.0,0,7,more_reliable,0.929,0.774,87.62
9,06037464200,2846,52.21,190991.0,0,18,more_reliable,0.987,0.480,87.61
10,06037403324,2944,45.87,114547.0,0,6,more_reliable,0.979,0.806,87.21


In [4]:
# Test how sensitive the ranking is to different weights

weight_scenarios = {
    "demand_focused": {
        "demand": 0.70,
        "income": 0.10,
        "competition": 0.20
    },
    "balanced": {
        "demand": 0.60,
        "income": 0.20,
        "competition": 0.20
    },
    "competition_focused": {
        "demand": 0.50,
        "income": 0.15,
        "competition": 0.35
    }
}

for scenario_name, weights in weight_scenarios.items():
    score_column = f"{scenario_name}_score"
    rank_column = f"{scenario_name}_rank"

    eligible_tracts[score_column] = (
        100
        * (
            weights["demand"]
            * eligible_tracts["demand_component"]
            + weights["income"]
            * eligible_tracts["income_percentile"]
            + weights["competition"]
            * eligible_tracts["competition_component"]
        )
    ).round(2)

    eligible_tracts[rank_column] = (
        eligible_tracts[score_column]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

rank_columns = [
    "demand_focused_rank",
    "balanced_rank",
    "competition_focused_rank"
]

eligible_tracts["top_20_scenario_count"] = (
    eligible_tracts[rank_columns]
    .le(20)
    .sum(axis=1)
)

stable_candidates = (
    eligible_tracts.loc[
        eligible_tracts["top_20_scenario_count"] >= 2,
        [
            "GEOID",
            "chinese_total_estimate",
            "chinese_share_pct",
            "median_household_income",
            "competitors_within_1_mile",
            "competitors_within_3_miles",
            "chinese_estimate_reliability",
            "demand_focused_score",
            "balanced_score",
            "competition_focused_score",
            "demand_focused_rank",
            "balanced_rank",
            "competition_focused_rank",
            "top_20_scenario_count"
        ]
    ]
    .sort_values(
        [
            "top_20_scenario_count",
            "balanced_rank"
        ],
        ascending=[False, True]
    )
)

print(
    "Stable candidates appearing in at least "
    "two scenario top-20 lists:",
    len(stable_candidates)
)

display(stable_candidates)

Stable candidates appearing in at least two scenario top-20 lists: 20


,GEOID,chinese_total_estimate,chinese_share_pct,median_household_income,competitors_within_1_mile,competitors_within_3_miles,chinese_estimate_reliability,demand_focused_score,balanced_score,competition_focused_score,demand_focused_rank,balanced_rank,competition_focused_rank,top_20_scenario_count
1679,06037403407,1661,70.14,156552.0,0,6,more_reliable,92.06,91.01,89.19,2,1,2,3
1671,06037430400,1674,39.66,182292.0,0,8,more_reliable,89.66,89.53,86.71,4,2,4,3
239,06037403325,2735,58.12,111445.0,0,2,more_reliable,93.04,89.06,89.60,1,4,1,3
1657,06037403404,1035,48.96,173289.0,0,7,more_reliable,88.66,88.61,86.51,8,5,6,3
700,06037464101,1376,61.90,250001.0,0,13,more_reliable,88.03,88.52,82.98,11,6,19,3
1870,06037670413,881,17.99,214625.0,0,0,more_reliable,86.86,88.16,88.89,14,7,3,3
141,06037408503,1943,29.42,145921.0,0,7,more_reliable,88.72,87.62,85.84,7,8,9,3
1023,06037403324,2944,45.87,114547.0,0,6,more_reliable,90.81,87.21,86.42,3,10,7,3
172,06037408703,2644,49.64,127500.0,0,10,more_reliable,89.51,87.02,83.78,5,11,16,3
2055,06037670326,680,21.36,250001.0,0,3,more_reliable,84.93,86.66,86.57,17,13,5,3


In [6]:
# Prepare score fields for spatial export

score_columns = [
    "GEOID",
    "demand_component",
    "competition_component",
    "demand_focused_score",
    "balanced_score",
    "competition_focused_score",
    "demand_focused_rank",
    "balanced_rank",
    "competition_focused_rank",
    "top_20_scenario_count"
]

score_results = eligible_tracts[
    score_columns
].copy()

import geopandas as gpd

# Load the complete tract spatial input layer

tract_input_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_site_selection_inputs.gpkg"
)

tracts = gpd.read_file(
    tract_input_path,
    layer="tract_site_selection_inputs"
)

print("Spatial tract rows loaded:", len(tracts))
print("Spatial CRS:", tracts.crs)

# Join scores to the complete tract geometry layer

scored_tracts = tracts.merge(
    score_results,
    on="GEOID",
    how="left",
    validate="one_to_one"
)

scored_tracts["screening_eligible"] = (
    scored_tracts["balanced_score"].notna()
)

scored_tracts["top_20_scenario_count"] = (
    scored_tracts["top_20_scenario_count"]
    .fillna(0)
    .astype(int)
)

print("All tract rows:", len(scored_tracts))

print(
    "Eligible scored tracts:",
    scored_tracts["screening_eligible"].sum()
)

print(
    "Stable candidates in at least two scenarios:",
    (
        scored_tracts["top_20_scenario_count"] >= 2
    ).sum()
)

# Save a tabular scoring output
score_csv_path = (
    project_root
    / "data"
    / "processed"
    / "la_preliminary_opportunity_scores.csv"
)

scored_tracts.drop(
    columns="geometry"
).to_csv(
    score_csv_path,
    index=False
)

# Save the spatial scoring layer
score_gpkg_path = (
    project_root
    / "data"
    / "processed"
    / "la_preliminary_opportunity_screening.gpkg"
)

if score_gpkg_path.exists():
    score_gpkg_path.unlink()

scored_tracts.to_file(
    score_gpkg_path,
    layer="preliminary_opportunity_screening",
    driver="GPKG",
    index=False
)

print("\nScore CSV:", score_csv_path)
print("Score GeoPackage:", score_gpkg_path)

Spatial tract rows loaded: 2498
Spatial CRS: EPSG:4269
All tract rows: 2498
Eligible scored tracts: 964
Stable candidates in at least two scenarios: 20

Score CSV: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_preliminary_opportunity_scores.csv
Score GeoPackage: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_preliminary_opportunity_screening.gpkg
